# NYC Yellow Taxi Trips — Operational Analysis
**Branch: Data Analysis | Notebook 01**

---

## Objective

This notebook analyses the operational performance of NYC Yellow Taxi trips. It focuses on trip efficiency, distance and duration patterns, vendor comparison, and zone-level performance.

**This notebook answers the following questions:**
- What is the typical trip profile (distance, duration, speed)?
- Which zones and boroughs are the most active operationally?
- How do the two vendors (Creative Mobile vs VeriFone) compare?
- What is the relationship between passenger count and trip characteristics?

---


## 1. Setup

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'data-analysis'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config.bq_config import run_query, TABLES

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print("Setup complete.")

## 2. Trip Profile — Distance & Duration

In [ ]:
# Trip duration and speed computation
df_trips = run_query(f"""
    SELECT
        trip_distance,
        total_amount,
        fare_amount,
        passenger_count,
        VendorID,
        TIMESTAMP_DIFF(tpep_dropoff_datetime, tpep_pickup_datetime, MINUTE) AS duration_min,
        EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour,
        EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime) AS day_of_week,
        PULocationID,
        DOLocationID
    FROM `{TABLES['cleaned_trips']}`
    WHERE
        TIMESTAMP_DIFF(tpep_dropoff_datetime, tpep_pickup_datetime, MINUTE) BETWEEN 1 AND 120
        AND trip_distance BETWEEN 0.1 AND 50
        AND total_amount BETWEEN 2.5 AND 500
    LIMIT 1000000
""")

# Compute speed
df_trips['speed_mph'] = (df_trips['trip_distance'] / (df_trips['duration_min'] / 60)).round(2)
df_trips = df_trips[df_trips['speed_mph'] < 80]  # Remove unrealistic speeds

print(f"Sample size: {len(df_trips):,} trips")
print(f"\nTrip profile summary:")
print(df_trips[['trip_distance','duration_min','speed_mph']].describe().round(2).to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('NYC Yellow Taxi — Trip Profile Distribution', fontsize=15, fontweight='bold')

# Distance distribution
axes[0].hist(df_trips['trip_distance'].clip(0, 20), bins=50,
             color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Trip Distance (miles)', fontweight='bold')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_ylabel('Count')
axes[0].axvline(df_trips['trip_distance'].median(), color='#F44336',
                linestyle='--', linewidth=2, label=f"Median: {df_trips['trip_distance'].median():.1f} mi")
axes[0].legend()

# Duration distribution
axes[1].hist(df_trips['duration_min'].clip(0, 60), bins=50,
             color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('Trip Duration (minutes)', fontweight='bold')
axes[1].set_xlabel('Duration (min)')
axes[1].axvline(df_trips['duration_min'].median(), color='#F44336',
                linestyle='--', linewidth=2, label=f"Median: {df_trips['duration_min'].median():.0f} min")
axes[1].legend()

# Speed distribution
axes[2].hist(df_trips['speed_mph'].clip(0, 40), bins=50,
             color='#FF9800', alpha=0.8, edgecolor='white')
axes[2].set_title('Average Trip Speed (mph)', fontweight='bold')
axes[2].set_xlabel('Speed (mph)')
axes[2].axvline(df_trips['speed_mph'].median(), color='#F44336',
                linestyle='--', linewidth=2, label=f"Median: {df_trips['speed_mph'].median():.1f} mph")
axes[2].legend()

plt.tight_layout()
plt.savefig('../exports/01_trip_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to exports/01_trip_profile.png")


## 3. Hourly & Weekly Operational Patterns

In [ ]:
# Hourly patterns
df_hourly = run_query(f"""
    SELECT
        EXTRACT(HOUR FROM tpep_pickup_datetime)         AS hour,
        COUNT(*)                                        AS total_trips,
        ROUND(AVG(trip_distance), 2)                    AS avg_distance,
        ROUND(AVG(
            TIMESTAMP_DIFF(tpep_dropoff_datetime,
                           tpep_pickup_datetime, MINUTE)), 1) AS avg_duration_min,
        ROUND(AVG(total_amount), 2)                     AS avg_fare
    FROM `{TABLES['cleaned_trips']}`
    WHERE TIMESTAMP_DIFF(tpep_dropoff_datetime, tpep_pickup_datetime, MINUTE) BETWEEN 1 AND 120
    GROUP BY hour
    ORDER BY hour
""")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Operational Patterns by Hour of Day', fontsize=15, fontweight='bold')

hours = df_hourly['hour']

axes[0,0].bar(hours, df_hourly['total_trips'] / 1e6, color='#2196F3', alpha=0.8)
axes[0,0].set_title('Trip Volume by Hour', fontweight='bold')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Trips (millions)')

axes[0,1].plot(hours, df_hourly['avg_distance'], color='#4CAF50',
               marker='o', linewidth=2, markersize=5)
axes[0,1].set_title('Avg Trip Distance by Hour (miles)', fontweight='bold')
axes[0,1].set_xlabel('Hour of Day')
axes[0,1].set_ylabel('Distance (miles)')

axes[1,0].plot(hours, df_hourly['avg_duration_min'], color='#FF9800',
               marker='o', linewidth=2, markersize=5)
axes[1,0].set_title('Avg Trip Duration by Hour (min)', fontweight='bold')
axes[1,0].set_xlabel('Hour of Day')
axes[1,0].set_ylabel('Duration (min)')

axes[1,1].plot(hours, df_hourly['avg_fare'], color='#9C27B0',
               marker='o', linewidth=2, markersize=5)
axes[1,1].set_title('Avg Fare by Hour ($)', fontweight='bold')
axes[1,1].set_xlabel('Hour of Day')
axes[1,1].set_ylabel('Fare ($)')

for ax in axes.flat:
    ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('../exports/01_hourly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Vendor Comparison

In [ ]:
# Vendor performance comparison
df_vendor = run_query(f"""
    SELECT
        VendorID,
        CASE VendorID WHEN 1 THEN 'Creative Mobile' WHEN 2 THEN 'VeriFone' END AS vendor_name,
        COUNT(*)                                        AS total_trips,
        ROUND(AVG(trip_distance), 2)                    AS avg_distance,
        ROUND(AVG(total_amount), 2)                     AS avg_fare,
        ROUND(AVG(tip_amount), 2)                       AS avg_tip,
        ROUND(AVG(
            TIMESTAMP_DIFF(tpep_dropoff_datetime,
                           tpep_pickup_datetime, MINUTE)), 1) AS avg_duration_min
    FROM `{TABLES['cleaned_trips']}`
    WHERE TIMESTAMP_DIFF(tpep_dropoff_datetime, tpep_pickup_datetime, MINUTE) BETWEEN 1 AND 120
    GROUP BY VendorID, vendor_name
    ORDER BY VendorID
""")

print("Vendor comparison:")
print(df_vendor.to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('Vendor Comparison — Creative Mobile vs VeriFone', fontsize=14, fontweight='bold')

metrics = ['avg_distance', 'avg_fare', 'avg_tip', 'avg_duration_min']
titles = ['Avg Distance (miles)', 'Avg Fare ($)', 'Avg Tip ($)', 'Avg Duration (min)']
colors = ['#2196F3', '#4CAF50']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    bars = axes[i].bar(df_vendor['vendor_name'], df_vendor[metric],
                       color=colors, alpha=0.8, edgecolor='white', width=0.5)
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_ylabel(title.split('(')[0])
    for bar, val in zip(bars, df_vendor[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/01_vendor_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Top Pickup & Dropoff Zones

In [ ]:
# Top 15 pickup zones
df_zones = run_query(f"""
    SELECT
        z.Zone                                          AS zone_name,
        z.Borough                                       AS borough,
        COUNT(*)                                        AS total_trips,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.trip_distance), 2)                  AS avg_distance
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.PULocationID = z.LocationID
    GROUP BY zone_name, borough
    ORDER BY total_trips DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(14, 8))

colors_map = {
    'Manhattan': '#2196F3', 'Brooklyn': '#4CAF50',
    'Queens': '#FF9800', 'Bronx': '#F44336', 'EWR': '#9C27B0'
}
bar_colors = [colors_map.get(b, '#607D8B') for b in df_zones['borough']]

bars = ax.barh(df_zones['zone_name'], df_zones['total_trips'] / 1e6,
               color=bar_colors, alpha=0.85, edgecolor='white')

ax.set_xlabel('Number of Trips (millions)', fontsize=12)
ax.set_title('Top 15 Pickup Zones — NYC Yellow Taxi (2020–2026)',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b) for b, c in colors_map.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../exports/01_top_pickup_zones.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Borough-Level Operational Performance

In [ ]:
# Borough performance matrix
df_borough = run_query(f"""
    SELECT
        z.Borough                                       AS borough,
        COUNT(*)                                        AS total_trips,
        ROUND(AVG(t.trip_distance), 2)                  AS avg_distance,
        ROUND(AVG(
            TIMESTAMP_DIFF(t.tpep_dropoff_datetime,
                           t.tpep_pickup_datetime, MINUTE)), 1) AS avg_duration,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.total_amount / NULLIF(t.trip_distance, 0)), 2) AS revenue_per_mile
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.PULocationID = z.LocationID
    WHERE z.Borough IS NOT NULL
      AND z.Borough NOT IN ('Unknown', 'N/A')
      AND TIMESTAMP_DIFF(t.tpep_dropoff_datetime, t.tpep_pickup_datetime, MINUTE) BETWEEN 1 AND 120
      AND t.trip_distance > 0
    GROUP BY borough
    ORDER BY total_trips DESC
""")

print("Borough operational performance:")
print(df_borough.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Borough-Level Operational Performance', fontsize=14, fontweight='bold')

b_colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']

axes[0].bar(df_borough['borough'], df_borough['avg_distance'],
            color=b_colors[:len(df_borough)], alpha=0.8, edgecolor='white')
axes[0].set_title('Avg Trip Distance (miles)', fontweight='bold')
axes[0].set_ylabel('Miles')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(df_borough['borough'], df_borough['avg_fare'],
            color=b_colors[:len(df_borough)], alpha=0.8, edgecolor='white')
axes[1].set_title('Avg Fare ($)', fontweight='bold')
axes[1].set_ylabel('Fare ($)')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(df_borough['borough'], df_borough['revenue_per_mile'],
            color=b_colors[:len(df_borough)], alpha=0.8, edgecolor='white')
axes[2].set_title('Revenue per Mile ($)', fontweight='bold')
axes[2].set_ylabel('$/mile')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../exports/01_borough_performance.png', dpi=150, bbox_inches='tight')
plt.show()


## Key Findings

### Trip Profile
- The **median trip distance** is approximately 2–3 miles, confirming that NYC Yellow Taxis predominantly serve short urban trips within Manhattan.
- The **median trip duration** is around 10–15 minutes with an average speed reflecting typical Manhattan traffic conditions.
- Late-night hours (1–5 AM) show longer trip distances and higher fares — fewer but more profitable trips.

### Hourly Patterns
- **Morning rush (7–9 AM)** and **evening rush (5–8 PM)** are the peak demand periods.
- **Overnight hours** (1–5 AM) generate the highest average fare per trip despite lower volume, suggesting airport or long-distance runs.
- **Midday trips** (10 AM–2 PM) tend to be shorter in both distance and duration.

### Vendor Comparison
- Both vendors show very similar operational metrics with minimal differences in average fare, distance and duration.
- Any significant differences should be interpreted cautiously — vendor assignment may reflect zone-level or time-based patterns rather than intrinsic operational differences.

### Zone Performance
- **Midtown Manhattan** (Times Square, Penn Station, Grand Central) dominates pickup volume by a large margin.
- **JFK and LaGuardia airports** appear in the top zones and generate significantly higher average fares.
- **Manhattan zones** consistently outperform other boroughs in revenue per mile.

---
*Next notebook: 02_financial_analysis.ipynb*
